In [ ]:
import pandas as pd
import numpy as np
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import tensorflow.keras as keras
from sklearn.preprocessing import StandardScaler
import time
from sklearn.utils.class_weight import compute_class_weight

Census Income Data (censusData.csv) This data set contains demographic and employment 
information from the 1994 U.S. Census. This will be used to predict whether an 
individual’s annual income exceeds $50,000. Your label column is income_binary, which 
contains two values: <=50K and >50K. This column is convered into a binary 
numeric label (for example, 0 and 1) during data preparation.

In [ ]:
census_filename = os.path.join(os.getcwd(), "data_capstone", "censusData.csv")
# Load chosen dataset and save it to df
df = pd.read_csv(census_filename)
df.head()

- Chosen dataset: censusData.csv 
- Label is income (represented by column ”income_binary”). Predicting if an individual’s 
annual income exceeds 50,000 
- Features to potentially use: occupation, work_class, age, education, race, sex_selfID, hours_per_week 
- This problem matters because predicting which individuals have an income more than 50,000 
can help identify individuals who fall below this threshold. If the government has information 
about which individuals are predicted to having less than 50,000 annual income, then they can 
distribute necessary resources for the family as well as oppurtunities to increase their income. For the 
individuals whose annual income exceeds 50,000, the government can identify what features (such as 
education or occupation) lead to a higher income. Consequently,there could be initiatives to better 
inform young job-seekers about how to do better in their fields and get a good salary.

In [ ]:
# Load chosen dataset and save it to df
df = pd.read_csv(census_filename)
df.head()

In [ ]:
#determine class imbalance
print(df['income_binary'].value_counts()) #count of values in each category
print(df['income_binary'].value_counts(normalize=True)*100) #percentage of each category

#visualize w histogram
sns.countplot(data=df, x='income_binary') #countplot used for single categorical variable
plt.xlabel('Income')
plt.ylabel('Count')
plt.title('Count of values for each income category')
plt.show()

In [ ]:
#pandas for further analyzing the data
display(df.dtypes)
display(df.describe())
display(df.describe(include='object'))

In [ ]:
#summary stats + visualizations to understand feature relevance to label
#occupation, workclass, age, education, race, sex_selfID, hours_per_week
categorical = ['occupation', 'workclass', 'education', 'race', 'sex_selfID']
for feature in categorical:
    #plt.figure(figsize=(10,4))
    sns.countplot(data=df, x=feature, hue='income_binary', order=df[feature].value_counts().index)
    plt.title(f'Income Distribution by {feature}')
    plt.show()

numerical = ['age', 'hours-per-week']
for feature in numerical:
    plt.figure(figsize=(8,4))
    sns.barplot(data=df, x='income_binary', y=feature)
    plt.title(f'{feature} Distribution by Income')
    plt.show

In [ ]:
#missing values
nan_count = np.sum(df.isnull(), axis=0)
nan_count

In [ ]:
#outliers
#age
Q1a = df['age'].quantile(0.25)
Q3a = df['age'].quantile(0.75)
IQRa = Q3a - Q1a

lb_a = Q1a-1.5 * IQRa #lower bound
ub_a = Q3a+1.5 * IQRa #upper bound
outliers_age = df[(df['age'] < lb_a) | (df['age'] > ub_a)]

#hours-per-week
Q1h = df['hours-per-week'].quantile(0.25)
Q3h = df['hours-per-week'].quantile(0.75)
IQRh = Q3h - Q1h

lb_h = Q1h-1.5 * IQRh #lower bound
ub_h = Q3h+1.5 * IQRh #upper bound
outliers_hours = df[(df['hours-per-week'] < lb_h) | (df['hours-per-week'] >ub_h)]

#print number of outliers for each numeric value
print(f"Total outliers in 'age': {len(outliers_age)}")
print(f"Total outliers in 'hours-per-week': {len(outliers_hours)}")

Overall findings from data exploration:

SUMMARY: There is a class imbalance where <=50K makes up 75.919044% of the data and 
income less than 50K makes up 24.080956% of the data. Individuals at 25th and 50th percentile work 
40 hrs/week and individuals at 75th percentile work 45 hrs/week. Based on the 
visualizations, indivduals working more hours per week are more likely to earn more 
than 50k. Finally ’capital-gain’ and ’capital-loss’ at 25th, 50th, and 75th percentiles 
all have values of 0. This means that a majority of the population doesn’t participate 
in investments. Thus, those with substantial grains in investments serves as an indicator 
of a higer income bracket.
ACTIONS: need to deal with class imbalance.

MISSING VALUES: There are missing values in the following features: age, workclass, occupation, 
hours per week, and native country. Out of these there are 3 categorical features, and we cannot 
deal with the missing values there as we cannot replace the values with an average. If we 
replaced it with the average of the one-hot encoding of the features, that would not make 
sense as decimal values would not mean anything for categorical values.
ACTIONS: replace missing values in ’age’ and ’hours-per-week’ with the mean of the respective 
featues (without the missing values)

OUTLIERS: Total outliers in ’age’: 143 Total outliers in ’hours-per-week’: 8913 These outliers 
might actually affect salary. As we can see in the graph of income vs hours-per-week, slightly 
more individuals who work more hours get paid more than 50k. Because these 8,913 data points real
patterns of whether an individual crosses that 50k income threshold, keeping them in the dataset
will prevent our model from underfitting. The same goes with the 143 age outliers, with such a 
large dataset, 143 values won’t force our model to make incorrect predictions, it would rather 
prevent it from underfiting.
ACTIONS: no outliers need to be dealt with.

FEATURES TO DROP: 1. ’education’ because it is redundant with ’education_num’. ’education_num’
is better because it has the number of education years whereas ’education’ has a mix of ’masters’
’bachelors’ etc. with # of years of study. Also, this feature has 16 unique values, so one hot
encoding would have a lot of categories to encode. 2. ’native-country’ because most values say
”United States” so a one hot encoding of this would create a lot of repeated values. Also, in 1994
immigration and work policies weren’t as strict so even if someone is native of another country,
they could earn the same as someone who is native from the US 3. ’fnlwgt’ represents the # of
people that the row’s profile represents, so it is not a direct characteristic that would directly 
affect individual income.
ACTIONS: drop the outlined features.

ETHICAL CONSIDERATIONS
Protected characteristics like race, gender, and socioeconomic status can introduce bias in the
model. In terms of gender, women were paid much less than men in 1994, so the model might learn
to treat the male gender as a predictor of income abot 50k. Also, race is a factor in determining pay due to potential workforce discrimination based on race. The model might learn to map minority
groups to lower income bracket, further expanding historical discrimination. 

A false positive that an individual earns more than 50k, when they actually do not, could exclude that individual from essential resources like housing assistance or social security benefits. A false negative that an individual earns less than 50k, when they earn more could lead to mis-categorization of someone needing financial resources when they actually don’t need them (this also takes away resources from individuals who do actually ern less than 50k).



In [ ]:
#preparing data

#deal with missing values
nan_count = np.sum(df.isnull(), axis = 0)
nan_count

condition = nan_count > 0#look for cols w missing vals
col_names = nan_count[condition].index
print(col_names)
nan_cols = list(col_names) #convert to list
print(nan_cols)

nan_col_types = df[nan_cols].dtypes
nan_col_types

#cant fill categorical values
df['age_na'] = df['age'].isnull()
df['hours-per-week_na'] = df['hours-per-week'].isnull()
df.head()

#print("row 654: " + str(df['age'][645])) #before fillna
mean_ages=df['age'].mean() #mean for all non-null values
#print("mean for all non-null values: " + str(mean_ages))
df['age'].fillna(value=mean_ages, inplace=True)
#print("row 654: " + str(df['age'][645])) #after fillna

mean_hours = df['hours-per-week'].mean()
df['hours-per-week'] = df['hours-per-week'].fillna(mean_hours)

In [ ]:
#drop redundant/unnecessary columns
df.drop(columns=['education', 'native-country', 'fnlwgt'], inplace=True)
df.drop(columns=['age_na', 'hours-per-week_na'], inplace=True)
df.columns

In [ ]:
#one hot encoding & map income_binary to 1 and 0
to_encode = ['workclass', 'occupation', 'marital-status', 'relationship','race', 'sex_selfID']
for colname in to_encode:
    df_encoded = pd.get_dummies(df[colname], prefix=colname + '_')
    df = df.join(df_encoded)

df.drop(columns=to_encode, inplace=True)

#map income_binary to 0 and 1
df['income_binary'] = df['income_binary'].map({'<=50K': 0, '>50K': 1})

# Check the conversion to make sure it worked perfectly
print(df['income_binary'].value_counts())

df.head()

DATA PREP SUMMARY

- Handled missing values by taking the average of the non-null values in each of the features. Then,
replaces the null values in the original feature with the average values.

- Dropped the following features: 
1. ’education’ because it is redundant with ’education_num’.
’education_num’ is better because it has the number of education years whereas ’education’ has a
mix of ’masters’ ’bachelors’ etc. with # of years of study. Also, this feature has 16 unique values, so one hot encoding would have a lot of categories to encode. 
2. ’native-country’ because most values
say ”United States” so a one hot encoding of this would create a lot of repeated values. Also, in
1994 immigration and work policies weren’t as strict so even if someone is native of another country,they could earn the same as someone who is native from the US 
3. ’fnlwgt’ represents the # of people that the row’s profile represents, so it is not a direct characteristic that would directly affect individual income.

- Applied one-hot encoding to features to make all the features numeric before applygin logistic
regression.

In [ ]:
#train, test, evaluate, and improve ML model

# Create labeled examples from the dataset
X = df.drop(columns=['income_binary'])
y=df['income_binary']

# Create training and test sets out of the labeled examples
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,stratify=y, random_state=42)

#train
lr_sk = LogisticRegression(C=10**10, max_iter=2000, random_state=67)
lr_sk.fit(X_train, y_train)

#test
y_pred = lr_sk.predict(X_test)

#evaluate
initial_acc = accuracy_score(y_test, y_pred)
initial_f1 = f1_score(y_test, y_pred, average='binary')
print(f"Initial accuracy: {initial_acc:.4f}")
print(f"Baseline f1 score: {initial_f1:.4f}" )

#Initial accuracy: 0.8549
#Baseline f1 score: 0.6713



In [ ]:
# Perform model selection through grid search cross-validation (GridSearchCV)
# to identify optimal hyperparameter values for model

param_grid={
'C': [0.001, 0.01, 0.1, 1, 10, 100],
'class_weight': ['balanced']
}

#w max_iter=2000 -> TOTAL NO. of ITERATIONS REACHED LIMIT
grid_search = GridSearchCV(
estimator = LogisticRegression(max_iter=2500, random_state=67),
param_grid=param_grid,
cv=5,
scoring='f1',
n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"hyperparameters: {grid_search.best_params_}")
print(f"best validation f1 score: {grid_search.best_score_:.4f}")

In [ ]:
# Train, test and evaluate a final version of model using the optimal hyperparameter values.

#final version w hyperparameters
final_model = grid_search.best_estimator_
y_pred_final = final_model.predict(X_test)

lr_accuracy_final = accuracy_score(y_test, y_pred_final)
lr_f1_final = f1_score(y_test, y_pred_final, average='binary')

print(f"final accuracy score: {lr_accuracy_final:.4f}")

In [ ]:
# Interpret model's outputs

# Retrieve model coefficients
coefficients = final_model.coef_[0]
feature_names = X.columns

# Organize into a scannable DataFrame sorted by absolute influence
coef_df = pd.DataFrame({
'Feature': feature_names,
'Coefficient': coefficients
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False).drop(columns=['Abs_Coefficient'])

# Display top 10 most influential features driving predictions
print("Top 10 Most Influential Features:")
print(coef_df.head(10).to_string(index=False))

# Plot the top feature impacts visually
plt.figure(figsize=(10, 6))
top_features = coef_df.head(10)
plt.barh(top_features['Feature'][::-1], top_features['Coefficient'][::-1],color='steelblue')
plt.xlabel('Coefficient Value (Log-Odds Impact)')
plt.title('Top 10 Feature Weights driving Income Class Predictions')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

MODEL REFLECTION
1. I chose a logistic regression because the dataset consisted of a mix of numarical features (age
hours per week, capital gains) and categorical features (occupation, relationship status, etc.). 
Logistic regression was efficient in this scenario because I was able to do one-hot encoding on the
categorical variables. Also, logistic regression allowed me to exatly measure the weight of individual features to understand which ones influenced the prediction the most. Finally, the dataset had  a strong 76 to 24 class imbalance since many individuals earned less than 50k. 

2. While addressing the class imbalance the final accuracy score dropped from 0.85...% to 0.81...%, but the F1 score increased from 0.6713 to 0.6840. This increased showed that the model selection process shifted the decision boundary to create a more balanced class for the smaller class in the dataset. 

3. I predicted the model’s output by calculating the coefficients of each of the efeatures. Then I created a graph of the top 10 most influential features that drove the income class predictions. In the context of the problem, positive coeffieints mean that the feature would pull the model towards predicting an income higher than 50k. For example, relationship_wife (+1.14) increases the likelihood of a higher-income prediction. Negative coefficients mean that the feature would pull the model towards predicting a lower income. For example, maritial-status__never-married (-1.32) reduces the probaility of a high-income prediction. 

4. Yes, there are ethical concerns to the model. The strong negative weights assigned to sex_selfID__Female (-2.19) and race__Black (-0.90)reflect historic discriminations against these groups. If the model was being deployed in a real-world setting, women and Black individuals would be harmed since the model might incorectly predict their incomes. Because of this, the model will generate more false negtives which would be harmful to the impacted groups in terms of income-based eligibility for resources, funds, and loans. Ultimately, this ethical concern is harmful because it would reinforce the historic biases these two groups have experienced.

NEURAL NETWORK

In [ ]:
# Scale data for the neural network

# Create the scaler
scaler = StandardScaler()
# Fit the scaler on the training data and transform the training data
X_train_scaled = scaler.fit_transform(X_train)
# Use the same scaler to transform the test data
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Get the number of features in training data
n_features = X_train_scaled.shape[1]
# Create the neural network model
nn_model = keras.Sequential()
# Create the input layer and add the input layer to the 'nn_model' object
input_layer = keras.layers.InputLayer(input_shape=(n_features,))
nn_model.add(input_layer)

# Create the hidden layers and add the hidden layers to the 'nn_model' object
# Decision: How many layers? How many units? What activation function?
# Add a comment explaining your choices.
# Used 3 layers to let the model move past the linear model without letting the model from overfitting to the data.
# Scaling from 64 units --> 32 units --> 16 units narrows down subsequent layers and forces the network to compress information
# ReLU allows for non-linearity needed to learn the complex relationship between features

hidden_layer_1 = keras.layers.Dense(units=64, activation='relu')
nn_model.add(hidden_layer_1)
hidden_layer_2 = keras.layers.Dense(units=32, activation='relu')
nn_model.add(hidden_layer_2)
hidden_layer_3 = keras.layers.Dense(units=16, activation='relu')
nn_model.add(hidden_layer_3)

# Create the output layer and add the output layer to the 'nn_model' object
# Use the correct number of units and activation function for binary classification

output_layer = keras.layers.Dense(units=1, activation='sigmoid')
nn_model.add(output_layer)

# Print a summary of your model
nn_model.summary()

In [ ]:
#DEFINE OPTIMIZATION FUNCTION

# Decision: What learning rate will you use? Add a comment explaining your 
# decision.
#Used a learning rate of 0.001, which is steps big enough to learn at a 
# reasonable pace, and small enough to not miss the optimal solution
#sgd_optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
sgd_optimizer = keras.optimizers.SGD(learning_rate=0.001)

#sgd_optimizer = keras.optimizers.Adam(learning_rate=0.001) #because was getting Loss=Nan with sgdOptimizer


#DEFINE LOSS FUNCTION

loss_fn = keras.losses.BinaryCrossentropy(from_logits=False)

In [ ]:
#COMPILE MODEL & FIT TO TRAINING DATA
#nn_model.compile(optimizer=sgd_optimizer, loss=loss_fn, metrics=['accuracy'])
nn_model.compile(optimizer=sgd_optimizer, loss='binary_crossentropy',metrics=['accuracy'])

class ProgBarLoggerNEpochs(keras.callbacks.Callback):
    def __init__(self, num_epochs: int, every_n: int = 50):
        self.num_epochs = num_epochs
        self.every_n = every_n

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            s = 'Epoch [{}/ {}]'.format(epoch + 1, self.num_epochs)
            logs_s = ['{}: {:.4f}'.format(k.capitalize(), v)
                for k, v in logs.items()]

            s_list = [s] + logs_s
            print(', '.join(s_list))


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weights = {
    int(classes[0]): float(weights[0]),
    int(classes[1]): float(weights[1])
}

In [ ]:
# Decision: How many epochs? Add a comment.
#100 epochs = 100 passes through the training data which give the 
# model ample opportunity to understand complex patterns
t0 = time.time() # start time
num_epochs = 100
history = nn_model.fit(
    X_train_scaled,
    y_train,
    epochs=num_epochs,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[ProgBarLoggerNEpochs(num_epochs, every_n=5)],
    verbose=0
)
t1 = time.time() # stop time

print('Elapsed time: %.2fs' % (t1-t0))

In [ ]:
#VISUALIZE TRAINING PERFORMANCE
# Plot training loss and validation loss over epochs
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Plot training accuracy and validation accuracy over epochs
acc_key = 'accuracy' if 'accuracy' in history.history else 'acc'
val_acc_key = 'val_accuracy' if 'val_accuracy' in history.history else 'val_acc'
plt.plot(history.history[acc_key], label='Training Accuracy')
plt.plot(history.history[val_acc_key], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
#EVALUATE MODEL'S PERFORMANCE ON TEST SET
# Generate predictions from your neural network using your scaled test data
# nn_model.predict() returns probabilities — apply a threshold of 0.5 to get class labels

y_prob_nn = nn_model.predict(X_test_scaled)
y_pred_nn = (y_prob_nn >= 0.5).astype(int) #threshold of 0.5 to get class labels

# Compute accuracy and F1 score for the neural network and print the results
nn_accuracy_final = accuracy_score(y_test, y_pred_nn)
nn_f1_final = f1_score(y_test, y_pred_nn, average='binary')

print(f"neural network final accuracy score: {nn_accuracy_final:.4f}")
print(f"neural network final F1 score: {nn_f1_final:.4f}")

NEURAL NETWORK REFLECTION
1. Designed a neural network with three hideen layers with 64, 32, and 16 neurons. Using 3 layers allows the model to learn the complex and non-linear relationships between the demographic features and the income label. Also, reducing the units in each layer forces the model to compress the learned information, which reduces the risk of overfitting. Used ReLU activation function because of its computation speed and lack of vanishing gradient problem. For the output layer, I used a sigmoid activiation function because it outputs a probability, which can be used as class predictions. For the learning rate I chose 0.001 for a SGD because this rate is small enough where steps are big enough to learn at a reasonable
pace, and small enough to not miss the optimal solution. Finally, training
with 100 epochs gives the network 100 passes through the training data
which give the model ample opportunity to understand complex patterns
However, the validation performance stabalized around 50 epochs, indicating that most of the useful learning had already occured. 
2. The training curves indicated that the model learned quickly during the first 20 epochs. During this time, both the trainign and validation loss decreased rapidly while the training and validation accuracly slowly increased. Both of these indicate that the network was learning patterns in the data. At around 40 epochs the validation loss stabalized at 0.39 and the training accuracy improved slowly while validation accuracy remained stable at ~0.81. These factors suggest that there is mild overfitting in the model since the model improved performance on the training data
more than the validation data. However, the gap between training and
validation data was small, so the model would still generalize well to unseen data. There was no evidence of underfitting because both the loss and accuracy improved sigificantly before stabalizing. 
3. The neural network had a acduracy of 0.8121, and a F1 score of 0.6870. These results were consistent with what the training curves showed. Since the validation accuracy stablized around 0.81, it was expected that the model would generalize well to the test data. The test performance matched the validation performance closely, confirming that the model didn’t over fit the training data significantly.

In [ ]:
#COMPARE LOGISTIC REGRESSION AND NEURAL NETWORK
#comparison using metric variables created earlier in the code
results = pd.DataFrame({
'Metric': ['Accuracy', 'F1 Score'],
'KNN Model': [lr_accuracy_final, lr_f1_final],
'Neural Network': [nn_accuracy_final, nn_f1_final]
})
print(results.to_string(index=False))

REFLECTION: COMPARISON OF THE TWO MODELS
1. The two models performed similarly, but the neural network had slightly better overall performance. The logistic regression model achieved an accuracy of 0.8101 and an F1 score of 0.6840, while the neural network
achieved an accuracy of 0.8121 and an F1 score of 0.6870. These indiate
that both models were good candidates for this dataset. 
2. Although the neural network performed slightly better, the added complexity did not come with added benefits. The neural network required more processing, feature scaling, selecting the learning rate and number of epochs, and designing the network architecture. It also required more training time than the logistic regres sion. Since the improvement in accuracy and F1 score is very small in the neural network, the logistic regresssion provides almost the same performance with less complexity. Thus, the added complexity was not justified for this problem. 

3. If I were presenting these results to the client, I would reccomend deploying the logistic regression model. Although the performance on the neural network was slightly better, it would not justify the computation costs, use of additional time, and reduced interpretability. Logistic
regression would allow the client to see which features influenced
the model the most through their coefficients. This would make it easier to
understand why an individual was predicted to have an income greater than. 50k. Furthermore, being able to see which features influenced the model the most (and in which direction) would help the client understand future predictions better because the dataset contains sensitive infromation such
as race and gender which could introduce bias in the model. Since incorrect
predicions could lead to loss of resources and aid which lower income
individuals deserved, having more interpretability is valuable in this
context. In conclusion, the logistic regression model offers a better
balance of performance, interpretability, ease of deployment, and complexity for this use case. 

4. Next time, I would experiment more with the learning rate and number of
epochs for the neural network to see how changing those hyperparameters
would affect the neural network’s performance in comparison to
the logistic regression model. For preprocessing, I would like to experiment with different methods of handling the class imbalance. As of
right now, the models may not be performing the best be-
cause of the class imbalance, so adjusting the way I handled class
imbalance for both models could improve performance. Finally, I would use confusion matrices to better understand the trade-offs between false positives and false negatives before reccomending the final model for deployment.